In [ ]:
import random
import bisect
import copy
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, f1_score

DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

TRAIN_SCENES = list(range(4, 19))
VAL_SCENES = [3]
TEST_SCENES = [1, 2]

In [ ]:
from scipy.ndimage import convolve
from sklearn.decomposition import KernelPCA
from sklearn.preprocessing import StandardScaler

# Discover and split the files
all_files = sorted(list(DATA_DIR.glob("*.mat")))
train_files, val_files, test_files = [], [], []

for f in all_files:
    if f.stem in ["GM01", "GM02"]:
        test_files.append(f)
    elif f.stem == "GM03":
        val_files.append(f)
    else:
        train_files.append(f)

# Laplacian noise estimator
def noise_sigma(x):
    mask = np.array([[1, -2, 1], 
                     [-2, 4, -2],
                     [1, -2, 1]], dtype=float)
    scores = []
    for b in range(x.shape[2]):
        total = 0.0
        for r in range(1, x.shape[0]-1, 64):
            end = min(r+64, x.shape[0]-1)
            stripe = x[r-1:end+1, :, b].astype(float)
            total += np.abs(convolve(stripe, mask)[1:-1, 1:-1]).sum()
        scores.append(total * np.sqrt(np.pi / 2) / (6*(x.shape[0]-2)*(x.shape[1]-2)))
    return np.asarray(scores)

# Fit the KPCA strictly on training data
def fit_preprocessor(train_files, noise_factor=0.5, fit_samples=256, components=8):
    rng = np.random.default_rng(42)
    scores, samples = [], []
    
    # Added tqdm here for the noise estimation and sampling loop
    for file_path in tqdm(train_files, desc="Fitting KPCA on train scenes"):
        mat_data = sio.loadmat(file_path)
        x = mat_data["img"]
        
        scores.append(noise_sigma(x))
        pixels = x.shape[0] * x.shape[1]
        n = min(pixels, int(np.ceil(fit_samples / len(train_files))))
        indices = rng.choice(pixels, n, replace=False)
        samples.append(x[indices // x.shape[1], indices % x.shape[1], :].astype(np.float64))
        
    sigma = np.mean(scores, axis=0)
    cutoff = noise_factor * sigma.mean()
    keep_bands = np.flatnonzero(sigma < cutoff)
    
    s = np.concatenate(samples)[:fit_samples, keep_bands]
    
    scaler = StandardScaler().fit(s)
    s_scaled = scaler.transform(s)
    
    gamma = 1 / len(keep_bands)
    kpca = KernelPCA(n_components=components, kernel='rbf', gamma=gamma, 
                     eigen_solver='arpack', random_state=42)
    z = kpca.fit_transform(s_scaled)
    post_scaler = StandardScaler().fit(z)
    
    return {
        "keep_bands": keep_bands,
        "scaler": scaler,
        "kpca": kpca,
        "post_scaler": post_scaler
    }

# Apply the fitted state
def apply_preprocessor(img_array, state):
    keep_bands = state["keep_bands"]
    img_clean = img_array[:, :, keep_bands]
    
    h, w, c = img_clean.shape
    img_reshaped = img_clean.reshape(h * w, c)
    
    a = state["scaler"].transform(img_reshaped)
    z = state["kpca"].transform(a)
    img_pca = state["post_scaler"].transform(z)
    
    return img_pca.reshape(h, w, state["kpca"].n_components).astype(np.float32)

# Generate the preprocessor state
preprocessor_state = fit_preprocessor(train_files)

In [ ]:
def build_cache(file_list, cache_dir, preprocessor_state, split_name):
    for file_path in tqdm(file_list, desc=f"Building {split_name} cache"):
        file_stem = file_path.stem
        pixel_cache = cache_dir / f"{file_stem}_pixels.npy"
        label_cache = cache_dir / f"{file_stem}_labels.npy"
        
        if not (pixel_cache.exists() and label_cache.exists()):
            mat_data = sio.loadmat(file_path)
            
            # Apply KPCA, preserving the (H, W, C) spatial dimensions
            img_processed = apply_preprocessor(mat_data["img"], preprocessor_state)
            gt_map = mat_data["map"].astype(np.int64)
            
            # Save the raw 2D spatial maps directly
            np.save(pixel_cache, img_processed)
            np.save(label_cache, gt_map)

print("Building caches...")
build_cache(train_files, CACHE_DIR, preprocessor_state, split_name="Train")
build_cache(val_files, CACHE_DIR, preprocessor_state, split_name="Val")
build_cache(test_files, CACHE_DIR, preprocessor_state, split_name="Test")

In [ ]:
class HyperspectralTileDataset(Dataset):
    def __init__(self, file_paths, cache_dir, tile_size=256, augment=False):
        self.tile_size = tile_size
        self.augment = augment
        self.arrays = []
        self.items = []
        
        for file_idx, file_path in enumerate(file_paths):
            pixel_cache = cache_dir / f"{file_path.stem}_pixels.npy"
            label_cache = cache_dir / f"{file_path.stem}_labels.npy"
            
            x = np.load(pixel_cache, mmap_mode='r')
            y = np.load(label_cache, mmap_mode='r')
            self.arrays.append((x, y))
            
            # Generate top-left coordinates for non-overlapping 256x256 tiles
            h, w = y.shape
            for r in range(0, h, tile_size):
                for c in range(0, w, tile_size):
                    self.items.append((file_idx, r, c))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        file_idx, r, c = self.items[idx]
        x, y = self.arrays[file_idx]
        
        # Extract the patch
        a = x[r:r+self.tile_size, c:c+self.tile_size]
        b = y[r:r+self.tile_size, c:c+self.tile_size]
        
        h, w = b.shape
        
        # Pad boundary tiles to guarantee consistent 256x256 tensors
        if h < self.tile_size or w < self.tile_size:
            a = np.pad(a, ((0, self.tile_size - h), (0, self.tile_size - w), (0, 0)), mode='edge')
            b = np.pad(b, ((0, self.tile_size - h), (0, self.tile_size - w)), constant_values=-1)
            
        if self.augment:
            k = random.randrange(4)
            a, b = np.rot90(a, k), np.rot90(b, k)
            if random.random() < 0.5:
                a, b = np.fliplr(a), np.fliplr(b)
                
        # Convert to PyTorch format (C, H, W)
        return torch.from_numpy(a.transpose(2, 0, 1).copy()), torch.from_numpy(b.copy()).long()

# Reduce batch size drastically from 1024 to 2, as 256x256 spatial tiles require much more memory
BATCH_SIZE = 2 

train_dataset = HyperspectralTileDataset(train_files, CACHE_DIR, tile_size=256, augment=True)
val_dataset = HyperspectralTileDataset(val_files, CACHE_DIR, tile_size=256, augment=False)
test_dataset = HyperspectralTileDataset(test_files, CACHE_DIR, tile_size=256, augment=False)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class DenseViT(nn.Module):
    def __init__(self, channels, tile=256, token=16, dim=32, depth=1, heads=4):
        super().__init__()
        if tile % token or dim % heads:
            raise ValueError('tile must divide by token and dim must divide by heads')
            
        self.tile = tile
        self.token = token
        
        # Extracts non-overlapping patches (16x16 pixels)
        self.embed = nn.Conv2d(channels, dim, token, stride=token)
        
        # Positional embeddings for the transformer sequence
        self.position = nn.Parameter(torch.zeros(1, (tile//token)**2, dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        
        # Transformer blocks
        block = nn.TransformerEncoderLayer(dim, heads, dim*4, dropout=0.1,
                                           batch_first=True, norm_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(block, depth, enable_nested_tensor=False)
        
        # Projects each transformer token back out to its constituent pixels
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, token*token))

    def forward(self, x):
        z = self.embed(x).flatten(2).transpose(1, 2) + self.position
        z = self.head(self.encoder(z))
        g = self.tile // self.token
        
        # Reshape back to the original 2D spatial format (Batch, Tile, Tile)
        return z.reshape(-1, g, g, self.token, self.token).permute(0, 1, 3, 2, 4).reshape(-1, self.tile, self.tile)

In [ ]:
from torchinfo import summary

# Extract the input channel count from the KPCA preprocessor state dynamically
input_channels = preprocessor_state["kpca"].n_components

# Initialize using her compact hyperparameters: dim=32, depth=1, heads=4
model = DenseViT(channels=input_channels, tile=256, token=16, dim=32, depth=1, heads=4).to(device)

summary(model, input_size=[BATCH_SIZE, input_channels, 256, 256])

In [ ]:
def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    
    return {
        "AUC": float(roc_auc_score(y_true, y_prob)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "F1-Score": float(f1_score(y_true, y_pred, zero_division=0))
    }

In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()
    train_loss = 0.0 
    steps = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        
        # valid acts as a mask to ignore the -1 padded borders
        valid = y >= 0
        if not valid.any():
            continue

        optimizer.zero_grad(set_to_none=True)
        
        # Slicing with [valid] flattens only the real pixels into 1D arrays
        logits = model(X)[valid]
        targets = y[valid].float()
        
        # Standard PyTorch loss implementation
        loss = loss_fn(logits, targets)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        steps += 1

    return train_loss / max(steps, 1)

In [ ]:
def test_step(model, dataloader, loss_fn, device, threshold=0.5):
    model.eval()
    test_loss = 0.0
    steps = 0
    all_labels, all_probs = [], []

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            
            valid = y >= 0
            if not valid.any():
                continue
                
            logits = model(X)[valid]
            targets = y[valid].float()
            
            loss = loss_fn(logits, targets)
            test_loss += loss.item()
            steps += 1
            
            all_labels.append(targets.cpu().numpy())
            all_probs.append(logits.sigmoid().cpu().numpy())
                    
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    
    rates = calculate_metrics(all_labels, all_probs, threshold)
    return (test_loss / max(steps, 1)), rates

In [ ]:
# Calculate global class counts from the training dataset arrays to balance the loss
total_water = sum(int((y == 0).sum()) for _, y in train_dataset.arrays)
total_oil = sum(int((y == 1).sum()) for _, y in train_dataset.arrays)

# pos_weight applies a multiplier to the rare class (oil) without modifying data loops
pos_weight_value = total_water / max(total_oil, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

# Initialize PyTorch objects natively handling the imbalance
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.01)

def train(model, train_loader, val_loader, optimizer, loss_fn, epochs, device, threshold=0.5):
    best_f1 = -1.0
    best_model_wts = copy.deepcopy(model.state_dict())
    results = {"train_loss": [], "val_loss": [], "F1": [], "AUC": []}

    for epoch in tqdm(range(epochs), desc="Training Epochs"):
        train_loss = train_step(model, train_loader, loss_fn, optimizer, device)
        val_loss, rates = test_step(model, val_loader, loss_fn, device, threshold)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"         AUC: {rates['AUC']:.4f} | Precision: {rates['Precision']:.4f} | "
              f"Recall: {rates['Recall']:.4f} | Acc: {rates['Accuracy']:.4f} | F1: {rates['F1-Score']:.4f}")

        # Track best model based on Validation F1-Score
        if rates["F1-Score"] > best_f1:
            best_f1 = rates["F1-Score"]
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, DATA_DIR / "best_model.pt")

        results["train_loss"].append(train_loss)
        results["val_loss"].append(val_loss)
        results["F1"].append(rates["F1-Score"])
        results["AUC"].append(rates["AUC"])

    # Restore best weights before returning
    model.load_state_dict(best_model_wts)
    return model, results

# Execute training pipeline
NUM_EPOCHS = 10
THRESHOLD = 0.5

print(f"Starting training on {device} for {NUM_EPOCHS} epochs...")
best_model, history = train(
    model=model,
    train_loader=train_dataloader,
    val_loader=val_dataloader, 
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    device=device,
    threshold=THRESHOLD
)
print("Training complete! Best model weights saved and restored.")

In [ ]:
def get_enhanced_rgb(img_array, rgb_bands=[29, 19, 9]):
    """Extracts and enhances RGB bands (~650nm, ~550nm, ~480nm) from raw data."""
    rgb_raw = img_array[:, :, rgb_bands].astype(np.float32)
    rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)
    
    rgb_enhanced = np.zeros_like(rgb_clean)
    for c in range(3):
        channel = rgb_clean[:, :, c]
        p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
        rgb_enhanced[:, :, c] = np.clip((channel - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
    return rgb_enhanced

def visualize_full_scene(model, file_path, preprocessor_state, device, threshold=0.5, tile=256):
    # Load raw .mat data
    mat_data = sio.loadmat(file_path)
    img = mat_data["img"]
    gt_map = mat_data["map"]
    h, w = gt_map.shape
    
    # Generate RGB visual from raw uncompressed data
    rgb_enhanced = get_enhanced_rgb(img)
    
    # Process the image features for the model using your fitted KPCA state
    x_processed = apply_preprocessor(img, preprocessor_state)
    
    predictions = np.zeros((h, w))
    model.eval()
    
    # Traverse the image in 256x256 non-overlapping patches
    with torch.inference_mode():
        for r in range(0, h, tile):
            for c in range(0, w, tile):
                a = x_processed[r:r+tile, c:c+tile]
                patch_h, patch_w = a.shape[:2]
                
                # Pad boundary tiles to guarantee exact 256x256 dimensions
                if patch_h < tile or patch_w < tile:
                    a = np.pad(a, ((0, tile - patch_h), (0, tile - patch_w), (0, 0)), mode='edge')
                    
                a_tensor = torch.from_numpy(a.transpose(2, 0, 1)).unsqueeze(0).to(device)
                
                # Predict and threshold
                prob = model(a_tensor).sigmoid().squeeze(0).cpu().numpy()
                pred_binary = (prob >= threshold).astype(int)
                
                # Crop away any padding and place back into the full-resolution map
                predictions[r:r+patch_h, c:c+patch_w] = pred_binary[:patch_h, :patch_w]

    # Plot the results side-by-side
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    
    axs[0].imshow(rgb_enhanced)
    axs[0].set_title(f"Actual Data (RGB Enhanced) - {file_path.stem}")
    axs[0].axis("off")
    
    axs[1].imshow(gt_map, cmap="inferno")
    axs[1].set_title("Ground Truth Mask")
    axs[1].axis("off")
    
    axs[2].imshow(predictions, cmap="inferno")
    axs[2].set_title(f"Model Prediction (Threshold={threshold})")
    axs[2].axis("off")
    
    plt.tight_layout()
    plt.show()

# Run visualization on the first test scene (e.g., GM01.mat)
visualize_full_scene(best_model, test_files[0], preprocessor_state, device, THRESHOLD)

In [ ]:
import numpy as np
import scipy.io as sio
from scipy.ndimage import binary_dilation
from pathlib import Path

# Setup paths based on your notebook structure
DATA_DIR = Path.cwd().parent / "data" / "hyperspectral_oil_spill"
train_files = [f for f in DATA_DIR.glob("*.mat") if f.stem not in ["GM01", "GM02"]]

total_oil = 0
total_hard_water = 0
total_easy_water = 0

print(f"{'Scene':<10} | {'Oil Pixels':<12} | {'Hard Water (Halo)':<18} | {'Easy Water (Open)':<18}")
print("-" * 67)

for f in train_files:
    mat = sio.loadmat(f)
    gt = mat["map"].astype(np.int64)
    
    oil_mask = (gt == 1)
    water_mask = (gt == 0)
    
    n_oil = np.sum(oil_mask)
    if n_oil == 0:
        continue
        
    # Apply your exact dataset logic
    dilated_oil = binary_dilation(oil_mask, iterations=3)
    hard_water_mask = dilated_oil & water_mask
    easy_water_mask = water_mask & ~hard_water_mask
    
    n_hard = np.sum(hard_water_mask)
    n_easy = np.sum(easy_water_mask)
    
    total_oil += n_oil
    total_hard_water += n_hard
    total_easy_water += n_easy
    
    print(f"{f.stem:<10} | {n_oil:<12} | {n_hard:<18} | {n_easy:<18}")

print("-" * 67)
print(f"{'TOTAL':<10} | {total_oil:<12} | {total_hard_water:<18} | {total_easy_water:<18}")
print("\n--- Diagnostic Conclusion ---")
print("Because the dataset loader slices with `water_sampled = np.vstack(...)[:n_oil]`,")
print("if 'Hard Water' >= 'Oil Pixels' in a scene, the model saw exactly ZERO Easy Water patches.")